# 🎥 AI Video Translator - Google Colab Acceleration Notebook

Welcome! This notebook is designed to run the **AI Video Translator** pipeline using Google Colab's high-speed cloud GPU resources (such as NVIDIA T4, L4, or A100). This will dramatically accelerate the speech-to-text, translation, TTS dubbing, Wav2Lip syncing, and GFPGAN face enhancement processes.

## 🚀 How to use this with the VS Code Colab Extension:
1. Install the official **Google Colab** extension in VS Code.
2. Open this `.ipynb` file in VS Code.
3. Click **Select Kernel** in the top right corner of the notebook editor.
4. Select **Colab** and choose your desired GPU runtime.
5. Sign in to your Google Account when prompted.

## 🔧 Step 1: Verify GPU Acceleration
Run the cell below to check if a GPU is active and see what GPU model Google has allocated for your session.

In [ ]:
!nvidia-smi

: 

## 📁 Step 2: Upload Project Codebase
Since Google Colab runs in a fresh remote container, we need to upload the project files (`main.py`, the `src` folder, `Wav2Lip`, etc.).

### Instructions:
1. On your local machine, zip the project folder.
   * **IMPORTANT: Exclude large directories** like `.git`, `.venv`, `venv311`, `venv312`, `temp`, and `output` to keep the zip file small and upload times fast!
2. Run the cell below to open the upload prompt.
3. Select and upload your `.zip` file.
4. The cell will automatically unzip and organize the files in Colab.

In [ ]:
from google.colab import files
import zipfile
import os
import shutil

print("Please select and upload your project zip file:")
uploaded = files.upload()

zip_name = None
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        zip_name = filename
        break

if zip_name:
    print(f"Extracting {zip_name}...")
    # Extract zip
    with zipfile.ZipFile(zip_name, 'r') as zip_ref:
        zip_ref.extractall('.')
    
    # Clean up the zip file to save space
    os.remove(zip_name)
    print("Project files extracted successfully!")
    
    # If the zip contents were inside a subfolder, move them to root
    extracted_items = os.listdir('.')
    for item in extracted_items:
        if os.path.isdir(item) and any(f in os.listdir(item) for f in ['main.py', 'src', 'Wav2Lip']):
            print(f"Moving files out of subdirectory '{item}' to current directory...")
            for f in os.listdir(item):
                shutil.move(os.path.join(item, f), '.')
            os.rmdir(item)
            break
            
    !ls -la
else:
    print("No zip file uploaded! Please run this cell again.")

## 📦 Step 3: Install Dependencies
This cell will install the python libraries required to run the pipeline. Since Colab already has PyTorch and CUDA pre-installed, we will install the remaining packages.

In [ ]:
# Install dependencies from the requirements file
if os.path.exists("requirements.txt"):
    print("Installing requirements.txt dependencies...")
    !pip install -r requirements.txt
else:
    print("requirements.txt not found. Installing default packages...")
    !pip install librosa soundfile moviepy opencv-python tqdm openai-whisper deep-translator googletrans==4.0.0-rc1 python-dotenv pydantic

# Install Coqui TTS and GFPGAN specifically
print("Installing TTS, GFPGAN, and fallback translator backends...")
!pip install TTS gfpgan deep-translator

# Ensure Wav2Lip dependencies are installed
if os.path.exists("Wav2Lip/requirements.txt"):
    !pip install -r Wav2Lip/requirements.txt

# Verify installation of GPU-supported PyTorch
import torch
print("PyTorch version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Current GPU Device:", torch.cuda.get_device_name(0))

## 📥 Step 4: Download Model Checkpoints
Instead of uploading huge weights from your computer, we can pull them directly to Colab's cloud storage at 100+ MB/s.

In [ ]:
import urllib.request

os.makedirs("models", exist_ok=True)
os.makedirs("gfpgan/weights", exist_ok=True)

# 1. Download Wav2Lip GAN checkpoint if missing
wav2lip_checkpoint_path = "models/wav2lip_gan.pth"
if not os.path.exists(wav2lip_checkpoint_path):
    print("Downloading wav2lip_gan.pth model weights...")
    url = "https://huggingface.co/spaces/wav2lip/wav2lip/resolve/main/checkpoints/wav2lip_gan.pth"
    urllib.request.urlretrieve(url, wav2lip_checkpoint_path)
    print("Wav2Lip weights downloaded!")
else:
    print("Wav2Lip weights already present.")

# 2. Download GFPGAN weights if missing
gfpgan_checkpoint_path = "gfpgan/weights/GFPGANv1.4.pth"
if not os.path.exists(gfpgan_checkpoint_path):
    print("Downloading GFPGANv1.4 weights...")
    url = "https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth"
    urllib.request.urlretrieve(url, gfpgan_checkpoint_path)
    print("GFPGAN weights downloaded!")
else:
    print("GFPGAN weights already present.")

## 🎥 Step 5: Upload Input Video
Run this cell to upload the input video file you want to translate and dub.

In [ ]:
print("Upload your input video file:")
video_upload = files.upload()

input_video_name = None
for filename in video_upload.keys():
    input_video_name = filename
    break

if input_video_name:
    os.makedirs("input", exist_ok=True)
    target_input_path = os.path.join("input", input_video_name)
    shutil.move(input_video_name, target_input_path)
    print(f"Successfully uploaded and moved input video to: {target_input_path}")
else:
    print("No video file uploaded! Please run this cell again.")

## 🚀 Step 6: Run the Translation Pipeline
Configure your settings using the variables in the cell below, then run the cell to kick off the pipeline.

In [ ]:
# Set configuration parameters:
INPUT_VIDEO = f"input/{input_video_name}" if input_video_name else "input/example.mp4"
OUTPUT_VIDEO = "output/dubbed_output.mp4"
TARGET_LANGUAGE = "es"        # Change to target language code (e.g. 'es', 'fr', 'de', 'hi')
WHISPER_MODEL = "medium"      # Whisper size: 'tiny', 'base', 'small', 'medium', 'large'
ENHANCE = True                # Enable GFPGAN face restoration (high quality, works fast on GPU)

# Ensure output directory exists
os.makedirs("output", exist_ok=True)

# Construct the command line arguments
cmd = [
    "python", "main.py",
    "--input_video", INPUT_VIDEO,
    "--output_video", OUTPUT_VIDEO,
    "--checkpoint_path", "models/wav2lip_gan.pth",
    "--whisper_model", WHISPER_MODEL,
    "--output_language", TARGET_LANGUAGE,
]

if ENHANCE:
    cmd.append("--enhance")

# Execute command
cmd_str = " ".join(cmd)
print(f"Executing: {cmd_str}")
!{cmd_str}

## 📥 Step 7: Download the Dubbed Video
After the execution cell finishes, run the cell below to download the final translation back to your local computer.

In [ ]:
if os.path.exists(OUTPUT_VIDEO):
    print(f"Downloading {OUTPUT_VIDEO}...")
    files.download(OUTPUT_VIDEO)
else:
    print(f"Output video not found at: {OUTPUT_VIDEO}. Make sure the pipeline completed successfully.")